# Étude Approfondie : Qualité et Cohérence Codes Géographiques

## 1. Introduction et Objectifs

Le succès de ce projet repose sur la capacité à fusionner trois bases de données différentes (Revenus, IRVE, Immatriculations). Le seul point commun entre ces fichiers est le Code INSEE de la commune.

Objectifs de ce notebook :
- Vérifier l'intégrité des codes INSEE dans chaque dataset.
- Identifier et corriger les anomalies (formats numériques, codes postaux confondus avec codes INSEE).
- Valider la capacité de jointure (unicité et complétude).

## 2. Initialisation et Diagnostic

On commence par charger nos outils et faire un état des lieux rapide des colonnes qui nous serviront de clés.

In [ ]:
import pandas as pd
from src.loading import load_irve_data, load_revenu_data, load_all_datasets
from src.utils import diagnostic_cle_jointure

# Chargement des bases brutes
PATHS = {
    'irve': 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435',
    'revenu': 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',
    've': 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
}
df_irve, df_revenu, df_ve = load_all_datasets(PATHS)

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenu.columns)

Nous remarquons que ces 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

Effectuons un diagnostic sur ces variables.

In [ ]:
from src.utils import diagnostic_cle_jointure
diagnostic_cle_jointure(df_irve,"code_insee_commune","df_irve")
print("")
diagnostic_cle_jointure(df_ve,"CODGEO","df_ve")
print("")
diagnostic_cle_jointure(df_revenu,"Code géographique","df_revenus")

Ce diagnostic met en évidence plusieurs points d’attention :

- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

## 3. Standardisation des Codes

#### Repérons les problèmes de longueurs de codes incohérentes

In [ ]:
# remplacer le 7 par 3 et 6
taile_louche = df_irve[
    df_irve["code_insee_commune"].notna() &
    (df_irve["code_insee_commune"].astype(str).str.len() == 7)
]
taile_louche["code_insee_commune"].sample(5)

Les valeurs des `code_insee_commune`de longueur 3 sont : "0.0".
Les valeurs des `code_insee_commune`de longueur 6 sont les codes à 4 chiffres suivis de ".0".
Les valeurs des `code_insee_commune`de longueur 7 sont les codes à 5 chiffres suivis de ".0".

In [ ]:
taile_louche = df_ve[
    df_ve["CODGEO"].notna() &
    (df_ve["CODGEO"].astype(str).str.len() == 4)
]
taile_louche["CODGEO"].sample(5)

Les valeurs des `code_insee_commune`de longueur 4 sont les codes à 4 chiffres des départements 1 (01) à 9 (09). Il faut rajouter un "0" au début du code.

#### Uniformisons ces codes géographiques pour qu'ils soient tous à 5 chiffres

In [ ]:
from src.cleaning import nettoyer_code_insee

df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
df_ve["CODGEO"] = df_ve["CODGEO"].apply(nettoyer_code_insee)
df_revenu["Code géographique"] = df_revenu["Code géographique"].apply(nettoyer_code_insee)

print("Vérification après nettoyage :")
print(f"Longueurs dans IRVE : {df_irve['code_insee_commune'].str.len().unique()}")
print(f"Longueurs dans VE : {df_ve['CODGEO'].str.len().unique()}")

## 4. Résolution des Conflits Géographiques (Cas de l'IRVE)

La base IRVE présente une part significative de valeurs manquantes pour les codes INSEE. Afin de maximiser le taux de recouvrement lors de la jointure, nous procédons à un géocodage inverse en utilisant les coordonnées géographiques (latitude et longitude) pour identifier la commune de rattachement de chaque point de charge.

In [ ]:
from src.loading import charger_communes
from src.utils import creer_gdf_irve, joindre_communes, ajouter_codes_geo

gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result,"both")


colonnes_irve = ["code_insee_commune", "code_geo_manquant", "code_geo_total"]

# Valeurs manquantes
print("Valeurs manquantes :")
print({col: df_irve[col].isna().sum() for col in colonnes_irve})

# Valeurs uniques
print("\nValeurs uniques :")
print({col: df_irve[col].nunique() for col in colonnes_irve})

Nous avons maintenant 3 colonnes concernant le code géographique dans la base de données irve :
- `code_insee_commune` : la variable initiale
- `code_geo_manquant` : la variable initiale complétée grace à la latitude et la longitude pour les valeurs manquantes
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude, sans prendre en compte la variable initiale `code_insee_commune`

Ces 2 méthodes permettent de diminuer nettement le nombre de valeurs manquantes.

In [ ]:
from src.utils import analyser_recouvrement_cles

set_irve_manquant = set(df_irve["code_geo_manquant"].dropna())
set_irve_total = set(df_irve["code_geo_total"].dropna())
set_ve = set(df_ve["CODGEO"].dropna())
set_revenus = set(df_revenu["Code géographique"].dropna())

analyser_recouvrement_cles(set_irve_manquant,set_ve,"code_geo_manquant","VE")
analyser_recouvrement_cles(set_irve_total,set_ve, "code_geo_total", "VE")

analyser_recouvrement_cles(set_irve_manquant,set_revenus,"code_geo_manquant","Revenu")
analyser_recouvrement_cles(set_irve_total,set_revenus, "code_geo_total", "Revenu")

codes_manquants_communs = (set_irve_manquant - set_revenus) & (set_irve_manquant - set_ve)
print("Nombre de codes de code_geo_manquant manquants à la fois dans la base Revenu et la base Véhicules Électriques:")
print(len(codes_manquants_communs))

L'analyse des ensembles de codes (sets) révèle un phénomène intéressant sur la qualité des données sources.

Les bases Revenu et Véhicules Électriques sont des référentiels exhaustifs couvrant la quasi-totalité des communes françaises. Il est donc révélateur de constater que la variable code_geo_manquant contient 823 codes totalement absents de ces deux bases.

Ce décalage peut être du à l'existence d'erreurs de saisie ou de codes obsolètes dans la base IRVE initiale. À l'inverse, notre variable recalculée code_geo_total s'aligne quasi parfaitement avec les référentiels nationaux (avec seulement 3 exceptions).

Le géocodage spatial ne se contente pas de remplir les trous, il semble rectifier les erreurs de localisation de la base source.

___________ début ___________

In [ ]:
from src.preparation_data import corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal

In [ ]:
df_irve = corriger_codes_incoherents(df_irve,codes_manquants_communs)

set_irve_manquant_final = set(df_irve["code_geo_manquant"].dropna())
nouveaux_manquants_communs = (set_irve_manquant_final - set_revenus) & (set_irve_manquant_final - set_ve)

print(f"Nombre de codes problématiques restants : {len(nouveaux_manquants_communs)}")

In [ ]:
df_irve = corriger_par_nom(df_irve, nouveaux_manquants_communs)

set_final = set(df_irve["code_geo_manquant"].dropna())
reste_a_corriger = (set_final - set_revenus) & (set_final - set_ve)
print(f"Nombre de codes problématiques restants après les deux étapes : {len(reste_a_corriger)}")

In [ ]:
df_irve["consolidated_code_postal"]=df_irve["consolidated_code_postal"].apply(nettoyer_code_insee)

vérifier que le code postal est bien de la même forme que le code insee

In [ ]:
corriger_conflit_code_postal(df_irve, reste_a_corriger)

set_anomalies_final = set(df_irve["code_geo_manquant"].dropna())
anomalies_finales = (set_anomalies_final - set_revenus) & (set_anomalies_final - set_ve)

print(f"Nombre final de codes géographiques non répertoriés : {len(anomalies_finales)}")


Pour les anomalies restantes je pense que c'est aussi le code postal qui est renseigné à la place du code géo, cela peut signifier que d'autres erreurs sont présentes dans "code_geo_manquant"...

___________ fin ___________